In [ ]:

!pip install spacy scikit-learn bertopic sentence-transformers gensim matplotlib plotly networkx
!python -m spacy download fr_core_news_sm

In [ ]:
import os
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import plotly.express as px
import networkx as nx

import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

In [ ]:
df = pd.read_csv("data/archelect_search.csv")

NM = "non mentionné"


df["parti"] = np.select(
    [
        df["titulaire-liste"] != NM,
        df["suppleant-liste"]  != NM,
        df["titulaire-soutien"] != NM,
        df["suppleant-soutien"] != NM,
    ],
    [
        df["titulaire-liste"],
        df["suppleant-liste"],
        df["titulaire-soutien"],
        df["suppleant-soutien"],
    ],
    default=NM
)

In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
df = df[df["date"].dt.year.isin([1973, 1978, 1981, 1988, 1993])]


## Chargement des textes

In [ ]:
TEXT_ROOT = "arkindex_archelec/text_files"

texts = {}
for dirpath, _, files in os.walk(TEXT_ROOT):
    for fname in files:
        if fname.endswith(".txt"):
            doc_id = fname.replace(".txt", "")
            text = open(os.path.join(dirpath, fname), encoding="utf-8", errors="ignore").read()
            texts[doc_id] = text

df["text"] = df["id"].map(texts)


In [ ]:
# Garder uniquement les lignes avec texte
df = df[df["text"].notna()].copy().reset_index(drop=True)
df["year"] = df["date"].dt.year

In [ ]:
# Nettoyage du header récurrent
df["text"] = df["text"].str.replace("Sciences Po / fonds CEVIPOF", "", regex=False)

## Lemmatisation

In [ ]:
def strip_accents(text):
    return unicodedata.normalize('NFD', text).encode('ascii', 'ignore').decode('ascii')

STOPWORDS = [x.strip() for x in open("data/stop_word_fr.txt").readlines()]
TOKEN_PATTERN = r"(?u)\b[a-zA-ZÀ-ÿ][a-zA-ZÀ-ÿ]+\b"


custom_stopwords = STOPWORDS + [
    "politique", "election", "electoral",
    "candidat", "candidature", "circonscription",
    "depute", "assemblee", "nationale",
    "vote", "voter", "scrutin",
    "parti","programme", "campagne",
    "janvier", "fevrier", "mars", "avril", "mai", "juin",
    "juillet", "aout", "septembre", "octobre", "novembre", "decembre",
    "france", "francais"
]


SAVE_PATH = "data/lemmatized.csv"
BATCH_SIZE = 2000

if os.path.exists(SAVE_PATH):
    done = pd.read_csv(SAVE_PATH)
    done_ids = set(done["id"])
    print(f"{len(done)} textes déjà lemmatisés, {len(df) - len(done)} restants")
else:
    done = pd.DataFrame(columns=["id", "lemmatized_text"])
    done_ids = set()
    print("Aucune sauvegarde trouvée, on repart de zéro")

remaining = df[~df["id"].isin(done_ids)][["id", "text"]].reset_index(drop=True)

if len(remaining) > 0:
    nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
    results = []
    for i, doc in enumerate(nlp.pipe(remaining["text"], batch_size=64)):
        results.append({
            "id": remaining.loc[i, "id"],
            "lemmatized_text": " ".join([strip_accents(token.lemma_) for token in doc])
        })
        if (i + 1) % BATCH_SIZE == 0:
            done = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
            done.to_csv(SAVE_PATH, index=False)
            results = []
            print(f"  {i + 1}/{len(remaining)} sauvegardés")
    if results:
        done = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
        done.to_csv(SAVE_PATH, index=False)
        print(f"  {len(remaining)}/{len(remaining)} sauvegardés")

df = df.merge(done[["id", "lemmatized_text"]], on="id", how="left")

## BERTopic

In [ ]:
docs = df["lemmatized_text"].fillna("").tolist()

# Charger les embeddings pré-calculés (ou recalculer si nécessaire)
#embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
#embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=64)
#np.save("data/BERTopic/embeddings.npy", embeddings)
embeddings = np.load("data/BERTopic/embeddings.npy")

In [ ]:
umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine', low_memory = False )

hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words=custom_stopwords,
    token_pattern=TOKEN_PATTERN
)



topic_model = BERTopic(
    #umap_model=umap_model,
    embedding_model=SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2"),
    #hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    nr_topics=11,
    verbose=True
)

In [ ]:
topics, probs = topic_model.fit_transform(docs, embeddings)
df["bertopic_topic"] = topics
topic_model.get_topic_info()

In [ ]:
# Préparer le corpus tokenisé (utilise tes docs lemmatisés)
tokenized = [doc.split() for doc in docs]
dictionary = Dictionary(tokenized)

# Extraire les top mots de chaque topic (hors outliers)
topics_words = [
    [w for w, _ in topic_model.get_topic(t)][:10]
    for t in topic_model.get_topics()
    if t != -1
]


# u_mass (rapide, pas besoin de corpus externe)
cm_umass = CoherenceModel(
    topics=topics_words,
    texts=tokenized,
    dictionary=dictionary,
    coherence='u_mass'
)
print(f"u_mass : {cm_umass.get_coherence():.4f}")

In [ ]:
cm_cv = CoherenceModel(
    topics=topics_words,
    texts=tokenized,
    dictionary=dictionary,
    coherence='c_v'
)
print(f"c_v    : {cm_cv.get_coherence():.4f}")

In [ ]:
topic_model.visualize_documents(
    docs=docs,
    embeddings=embeddings,
    hide_annotations=True,
    topics=[0, 1, 2, 3, 4, 5,6, 7, 8, 9],
    height=600,
    width=1000
)

In [ ]:
topic_model.visualize_barchart(
    n_words = 10, # Select the number of words to display per topic
    topics = [0,1,2,3,4], # Select specific topics to display
    # top_n_topics = 6, # Select the first n topics to display
    # height = 300, # Adjust the height of the plot
    # width = 800 # Adjust the width of the plot 
)

In [ ]:
# --- 1. Carte intertopique (bulles proportionnelles à la taille du topic) ---
topic_model.visualize_topics()

In [ ]:
# --- 3. Dendrogramme hiérarchique des topics ---
hierarchical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
# --- 5. Heatmap de similarité entre topics ---
topic_model.visualize_heatmap()

In [ ]:
# --- 6. Distribution thématique d'un document ---
# probs de HDBSCAN = confidence du cluster assigné seulement (pas une distribution complète)
# approximate_distribution() calcule une vraie distribution sur tous les topics
topic_distr, _ = topic_model.approximate_distribution(docs[:1], window=4, stride=1)
topic_model.visualize_distribution(topic_distr[0])

In [ ]:
topics_reduced = topic_model.reduce_outliers(
    documents = docs, 
    topics = topics, 
    probabilities = probs, 
    embeddings = embeddings,
    strategy="embeddings" 
)

In [ ]:
pd.Series(topics_reduced).value_counts().sort_index()

In [ ]:
df["topic"] = topics_reduced

In [ ]:
topic_labels = {
    0: "Pouvoir national et travail gauche",
    1: "Écologie et protection de la nature",
    2: "Droits des travailleurs et patronat",
    3: "Lutte ouvrière et syndicats",
    4: "Travail, bourgeoisie et socialisme",
    5: "Majorité républicaine et alliances",
    6: "européen, acceptation etillusion",
    7: "Criminalité et lois",
    8: "UND/WIF (Alsace)",
    9: "Gauche radicale",
    -1: "Outliers"
}

df["topic_label"] = df["topic"].map(topic_labels)

topic_model.set_topic_labels(topic_labels)


In [ ]:
# Calcul des distributions

#topic_distr, _ = topic_model.approximate_distribution(docs, window=4, stride=1)
#np.save("data/BERTopic/topic_distr.npy", topic_distr)


In [ ]:
topic_distr = np.load("data/BERTopic/topic_distr.npy")

distr_df = pd.DataFrame(topic_distr, columns=[topic_labels[i] for i in range(len(topic_labels)-1)])  # sans -1
distr_df["parti"] = df["parti"].values

# Agréger par parti (moyenne des distributions)
party_profiles = distr_df.groupby("parti").mean()
print(party_profiles.shape)  # [n_partis, n_topics]
party_profiles.head()


In [ ]:
# Réduction en 2D
pca = PCA(n_components=2)
coords = pca.fit_transform(party_profiles.values)

# Topic dominant par parti
dominant_topic = party_profiles.idxmax(axis=1)

# Palette couleurs
unique_topics = dominant_topic.unique()
palette = plt.cm.tab10(np.linspace(0, 1, len(unique_topics)))
color_map = dict(zip(unique_topics, palette))
colors = [color_map[t] for t in dominant_topic]

# Plot
fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=100, alpha=0.8)

for i, parti in enumerate(party_profiles.index):
    ax.annotate(parti, (coords[i, 0], coords[i, 1]), fontsize=7, alpha=0.85)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("Carte sémantique des partis politiques (1973–1993)")

# Légende
legend = [Patch(color=color_map[t], label=t) for t in unique_topics]
ax.legend(handles=legend, fontsize=7, loc="best", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.savefig("data/carte_semantique_partis.png", dpi=150)
plt.show()


In [ ]:
n_docs_by_parti = distr_df.groupby("parti").size()
TOP_N = 30  # ajuste selon le résultat

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=100, alpha=0.8)

for i, parti in enumerate(party_profiles.index):
    if n_docs_by_parti[parti] >= n_docs_by_parti.nlargest(TOP_N).min():
        ax.annotate(parti, (coords[i, 0], coords[i, 1]), fontsize=8)

plt.show()

In [ ]:
pca3 = PCA(n_components=3)
coords3 = pca3.fit_transform(party_profiles.values)

plot_df3 = pd.DataFrame({
    "x": coords3[:, 0],
    "y": coords3[:, 1],
    "z": coords3[:, 2],
    "parti": party_profiles.index,
    "topic": dominant_topic.values,
    "n_docs": distr_df.groupby("parti").size().values
})

fig = px.scatter_3d(
    plot_df3, x="x", y="y", z="z",
    color="topic",
    hover_name="parti",
    size="n_docs",
    title="Carte sémantique 3D des partis (1973–1993)",
    labels={
        "x": f"PC1 ({pca3.explained_variance_ratio_[0]:.1%})",
        "y": f"PC2 ({pca3.explained_variance_ratio_[1]:.1%})",
        "z": f"PC3 ({pca3.explained_variance_ratio_[2]:.1%})",
    }
)
fig.show()


In [ ]:
# Similarité cosine entre partis
sim_matrix = cosine_similarity(party_profiles.values)
partis = party_profiles.index.tolist()

# Construire le graphe
SEUIL = 0.8  # ajuste si trop dense ou trop sparse
G = nx.Graph()

# Ajouter tous les noeuds d'abord (y compris les partis sans arête)
for i, parti in enumerate(partis):
    G.add_node(parti, topic=dominant_topic.iloc[i], size=distr_df[distr_df["parti"] == parti].shape[0])

for i, p1 in enumerate(partis):
    for j, p2 in enumerate(partis):
        if i < j and sim_matrix[i, j] > SEUIL:
            G.add_edge(p1, p2, weight=sim_matrix[i, j])

# Plot
fig, ax = plt.subplots(figsize=(18, 14))

pos = nx.spring_layout(G, k=2, seed=42)
node_colors = [color_map.get(G.nodes[n]["topic"], "grey") for n in G.nodes]
node_sizes  = [max(G.nodes[n]["size"] * 2, 50) for n in G.nodes]
edge_weights = [G[u][v]["weight"] for u, v in G.edges]

nx.draw_networkx_edges(G, pos, alpha=0.3, width=[w*2 for w in edge_weights], ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)

ax.set_title("Réseau de similarité thématique entre partis (1973–1993)")
ax.axis("off")

legend = [Patch(color=color_map[t], label=t) for t in unique_topics]
ax.legend(handles=legend, fontsize=7, loc="upper left")

plt.tight_layout()
plt.savefig("data/reseau_partis.png", dpi=150)
plt.show()


In [ ]:
# Fusionner les distributions avec les métadonnées
meta_cols = ["year", "titulaire-sexe", "titulaire-age-tranche",
             "titulaire-mandat-passe", "departement-nom"]

analysis_df = distr_df.copy()
for col in meta_cols:
    analysis_df[col] = df[col].values

# topic_cols = uniquement les colonnes numériques (distributions)
topic_cols = [c for c in distr_df.columns
              if c != "parti" and pd.api.types.is_numeric_dtype(distr_df[c])]

print(f"{len(topic_cols)} topics : {topic_cols}")


In [ ]:
yearly = analysis_df.groupby("year")[topic_cols].mean()

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(yearly.T, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(yearly.index))); ax.set_xticklabels(yearly.index)
ax.set_yticks(range(len(topic_cols))); ax.set_yticklabels(topic_cols, fontsize=8)
ax.set_title("Distribution moyenne des topics par année")
plt.colorbar(im, ax=ax, label="poids moyen")
plt.tight_layout(); plt.show()


In [ ]:
gender_df = analysis_df[analysis_df["titulaire-sexe"] != "non mentionné"]
by_gender = gender_df.groupby("titulaire-sexe")[topic_cols].mean()

fig, ax = plt.subplots(figsize=(12, 3))
im = ax.imshow(by_gender, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(topic_cols))); ax.set_xticklabels(topic_cols, fontsize=8, rotation=45, ha="right")
ax.set_yticks(range(len(by_gender))); ax.set_yticklabels(by_gender.index)
ax.set_title("Distribution des topics par sexe")
plt.colorbar(im, ax=ax, label="poids moyen")
plt.tight_layout(); plt.show()


In [ ]:
AGE_ORDER = ["entre 20 et 29 ans", "entre 30 et 39 ans", "entre 40 et 49 ans",
             "entre 50 et 59 ans", "entre 60 et 69 ans", "70 ans et plus"]

age_df = analysis_df[analysis_df["titulaire-age-tranche"].isin(AGE_ORDER)]
by_age = age_df.groupby("titulaire-age-tranche")[topic_cols].mean().reindex(AGE_ORDER)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(by_age, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(topic_cols))); ax.set_xticklabels(topic_cols, fontsize=8, rotation=45, ha="right")
ax.set_yticks(range(len(AGE_ORDER))); ax.set_yticklabels(AGE_ORDER, fontsize=8)
ax.set_title("Distribution des topics par tranche d'âge")
plt.colorbar(im, ax=ax, label="poids moyen")
plt.tight_layout(); plt.show()


In [ ]:
def categorize_mandat(m):
    if "député sortant" in str(m): return "Député sortant"
    if "non mentionné" in str(m): return "Nouveau candidat"
    return "Autre mandat"

analysis_df["statut"] = analysis_df["titulaire-mandat-passe"].apply(categorize_mandat)
by_statut = analysis_df.groupby("statut")[topic_cols].mean()

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(by_statut, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(topic_cols))); ax.set_xticklabels(topic_cols, fontsize=8, rotation=45, ha="right")
ax.set_yticks(range(len(by_statut))); ax.set_yticklabels(by_statut.index)
ax.set_title("Distribution des topics : sortants vs nouveaux")
plt.colorbar(im, ax=ax, label="poids moyen")
plt.tight_layout(); plt.show()


In [ ]:
import plotly.express as px

by_dept = analysis_df.groupby("departement-nom")[topic_cols].mean()
by_dept["topic_dominant"] = by_dept.idxmax(axis=1)
by_dept["score_dominant"] = by_dept[topic_cols].max(axis=1)
by_dept = by_dept.reset_index()

# Nécessite le GeoJSON des départements français
geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/departements-version-simplifiee.geojson"
import urllib.request, json
with urllib.request.urlopen(geojson_url) as r:
    geojson = json.load(r)

# Associer les noms (le geojson utilise des noms normalisés)
fig = px.choropleth(
    by_dept,
    geojson=geojson,
    locations="departement-nom",
    featureidkey="properties.nom",
    color="topic_dominant",
    title="Topic dominant par département",
    hover_data=["score_dominant"]
)
fig.update_geos(fitbounds="locations", visible=False)
fig.show()


In [ ]:
from matplotlib.patches import FancyArrowPatch

by_statut = analysis_df.groupby("statut")[topic_cols].mean()
categories = topic_cols
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # boucle

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = ["steelblue", "tomato", "green"]
for (label, row), color in zip(by_statut.iterrows(), colors):
    values = row.tolist() + row.tolist()[:1]
    ax.plot(angles, values, label=label, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=7)
ax.set_title("Profil thématique : sortants vs nouveaux", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout(); plt.show()


In [ ]:
yearly = analysis_df.groupby("year")[topic_cols].mean()

fig, ax = plt.subplots(figsize=(12, 5))
for col in topic_cols:
    ax.plot(yearly.index, yearly[col], marker="o", label=col)
ax.set_title("Évolution des topics par année")
ax.set_xlabel("Année"); ax.set_ylabel("Poids moyen")
ax.legend(fontsize=7, bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.show()
